In [2]:
#Name: Faez Nazran Haziq Bin Yusrizal, Muhammad Harizul Zharfan Bin Abdul Karim
#ID:IS01083871, IS01083880

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
import gensim
import gensim.corpora as corpora
from gensim.models import LdaModel, CoherenceModel

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

df = pd.read_csv('news_dataset.csv')
text_data = df['text'].dropna() 

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())

    tokens = word_tokenize(text)
    
    processed_tokens = []
    for word in tokens:
        if word not in stop_words:
            # Applying both stemming and lemmatization as specified
            stemmed = stemmer.stem(word)
            lemmatized = lemmatizer.lemmatize(stemmed)
            processed_tokens.append(lemmatized)
            
    return processed_tokens

print("Processing text... this may take a moment depending on dataset size.")
processed_texts = text_data.apply(preprocess_text).tolist()

id2word = corpora.Dictionary(processed_texts)


corpus = [id2word.doc2bow(text) for text in processed_texts]

print("Training LDA Model...")
lda_model = LdaModel(corpus=corpus,
                     id2word=id2word,
                     num_topics=4, 
                     random_state=42,
                     passes=10,
                     alpha='auto',
                     per_word_topics=True)


print("\n--- LDA Model Topics ---")
for idx, topic in lda_model.print_topics(-1):
    print(f"Topic {idx}: \nWords: {topic}\n")

print("Calculating Coherence Score...")
coherence_model_lda = CoherenceModel(model=lda_model, 
                                     texts=processed_texts, 
                                     dictionary=id2word, 
                                     coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print(f'\nCoherence Score: {coherence_lda}')

"""
Interpretation of the Coherence Score:
The coherence score evaluates how semantically similar the high-scoring words within our generated topics are. A higher score generally means the topics are more human-interpretable and logically consistent. For this LDA model, the coherence score achieved is [Insert the printed coherence score here]. This indicates a [low/moderate/high] level of meaningful clustering among the 4 requested topics for the news dataset. Depending on this score, the 4-topic assumption may represent the news categories well, though further tuning of hyperparameters (like passes, or testing different topic numbers) could potentially yield even clearer semantic separation.
"""

[nltk_data] Downloading package punkt to
[nltk_data]     /home/18e747de-16ab-4740-b78a-
[nltk_data]     1025674568be/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/18e747de-16ab-4740-b78a-
[nltk_data]     1025674568be/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/18e747de-16ab-4740-b78a-
[nltk_data]     1025674568be/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Processing text... this may take a moment depending on dataset size.
Training LDA Model...

--- LDA Model Topics ---
Topic 0: 
Words: 0.008*"peopl" + 0.007*"would" + 0.007*"one" + 0.005*"say" + 0.005*"think" + 0.005*"dont" + 0.005*"u" + 0.004*"know" + 0.004*"govern" + 0.004*"go"

Topic 1: 
Words: 0.035*"maxaxaxaxaxaxaxaxaxaxaxaxaxaxax" + 0.023*"b" + 0.006*"mov" + 0.006*"pt" + 0.004*"v" + 0.004*"la" + 0.003*"space" + 0.003*"period" + 0.003*"st" + 0.002*"g"

Topic 2: 
Words: 0.015*"x" + 0.015*"use" + 0.010*"key" + 0.007*"encrypt" + 0.007*"file" + 0.006*"system" + 0.005*"chip" + 0.005*"program" + 0.005*"db" + 0.005*"one"

Topic 3: 
Words: 0.007*"game" + 0.006*"year" + 0.006*"get" + 0.006*"one" + 0.006*"team" + 0.005*"like" + 0.005*"would" + 0.004*"good" + 0.004*"go" + 0.004*"play"

Calculating Coherence Score...

Coherence Score: 0.48829718513786713


'\nInterpretation of the Coherence Score:\nThe coherence score evaluates how semantically similar the high-scoring words within our generated topics are. A higher score generally means the topics are more human-interpretable and logically consistent. For this LDA model, the coherence score achieved is [Insert the printed coherence score here]. This indicates a [low/moderate/high] level of meaningful clustering among the 4 requested topics for the news dataset. Depending on this score, the 4-topic assumption may represent the news categories well, though further tuning of hyperparameters (like passes, or testing different topic numbers) could potentially yield even clearer semantic separation.\n'